# Notebook 05 - Connectivity Experiments

**Ramdas Chaugale - 40494109**  
**ECS8056 Themed Research Project**

A laser lesion is one connected region, but a segmentation network has no reason to produce one.
This notebook asks where that constraint belongs: inside the loss function, or after training.

The comparison is controlled. Hybrid U-Net and Soft Erosion Hybrid are the same architecture with
**zero** parameters between them: only the loss differs, so any difference in fragmentation is
attributable to the objective. Both input sizes are examined, because the erosion kernel scales
with the image and that turns out to matter.

Notebooks run in order: 01 → 02 → 03 → 04 → 05.

## 1. Setup

Paths resolve from the project root. Both runs are described by a dictionary carrying its input
size, checkpoint folder and the two kernels that scale with geometry.

In [ ]:
import os
import sys
import pickle
import time

import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn.functional as F
from PIL import Image

PROJECT_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))

DATA_DIR        = os.path.join(PROJECT_ROOT, 'data')
ENDPOINT_FOLDER = os.path.join(DATA_DIR, 'endpoint')
ANNO_FOLDER     = os.path.join(DATA_DIR, 'annotated',
                               'endpoint_annotated')
MASK_FOLDER     = os.path.join(DATA_DIR, 'masks')

OCTF_CODE      = os.path.join(PROJECT_ROOT, 'reconstruction',
                              'OCT-F_Reconstruction', 'code')
FINETUNE_CKPT  = os.path.join(PROJECT_ROOT, 'reconstruction',
                              'fine_tuned_reconstruction.pth')
RESIDUAL_CACHE = os.path.join(PROJECT_ROOT, 'reconstruction', 'residual_cache.pkl')

IMAGES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'images')
os.makedirs(IMAGES_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
# architectures.py is the same module the training run imported on the HPC.
# it is in code/ in the working copy and in src/ in the repository
for candidate in [os.path.join(PROJECT_ROOT, 'code'), os.path.join(PROJECT_ROOT, 'src')]:
    if os.path.exists(os.path.join(candidate, 'architectures.py')):
        sys.path.insert(0, candidate)
        break
sys.path.insert(0, OCTF_CODE)

SEED = 40494109
np.random.seed(SEED)
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 150, 'font.size': 11,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'legend.frameon': False,
    'figure.titlesize': 14, 'figure.titleweight': 'bold',
})

PRED_RGB = [255, 80, 80]
SMOOTH = 1.0

print(f"Seed: {SEED}")
print(f"Device: {device}")

In [ ]:
# one descriptor per run; the erosion kernel scales with the image,
# 5 at 256 and 9 at native, so it covers the same physical extent
RUN_256 = {'label': '256x256', 'suffix': '_256', 'size': (256, 256),
           'pp_kernel': 15, 'erosion_kernel': 5,
           'ckpt_dir': os.path.join(PROJECT_ROOT, 'reconstruction', 'checkpoints_256')}

RUN_NATIVE = {'label': '496x768', 'suffix': '', 'size': (496, 768),
              'pp_kernel': 29, 'erosion_kernel': 9,
              'ckpt_dir': os.path.join(PROJECT_ROOT, 'reconstruction', 'checkpoints')}

for run in [RUN_256, RUN_NATIVE]:
    assert os.path.exists(run['ckpt_dir']), f"MISSING: {run['ckpt_dir']}"
    print(f"{run['label']:<10} erosion kernel {run['erosion_kernel']:<3} "
          f"closing kernel {run['pp_kernel']}")

Both kernels scale with input geometry so that morphological operations span an equivalent
physical extent at either size: closing 15 and erosion 5 at 256, closing 29 and erosion 9 at
native.

## 2. Differentiability of a component-count objective

The constraint we want is "one connected region". The direct way to express that is to count
components and penalise anything above one. This section shows why that cannot be trained, and what
can be.

In [ ]:
torch.manual_seed(SEED)

# a prediction that carries gradients, as it would during training
prediction = torch.rand(1, 1, 64, 64, requires_grad=True)

binary = (prediction.detach().numpy()[0, 0] > 0.5).astype(np.uint8)
n_components = cv2.connectedComponents(binary)[0] - 1
cv2_loss = float(n_components)

print("Attempt 1: count components with OpenCV")
print(f"  components found: {n_components}")
print(f"  loss value: {cv2_loss}  (type {type(cv2_loss).__name__})")
print(f"  is it a tensor? {isinstance(cv2_loss, torch.Tensor)}")
print("  .backward() cannot be called on a float, so no gradient reaches the model.")

`cv2.connectedComponents` returns a Python float. Converting a tensor to a NumPy array and then
counting labels breaks the computation graph twice over, so there is no path back to the model
weights. This is not a tuning problem: the objective cannot be optimised by gradient descent at
all.

In [ ]:
prediction2 = torch.rand(1, 1, 64, 64, requires_grad=True)

# erosion is a minimum filter, and min(x) = -max(-x), so max pooling gives a
# differentiable erosion
eroded = -F.max_pool2d(-prediction2, kernel_size=9, stride=1, padding=4)
erosion_loss = (prediction2 - eroded).mean()
erosion_loss.backward()

print("Attempt 2: soft erosion via negated max pooling")
print(f"  loss value: {erosion_loss.item():.6f}  (type {type(erosion_loss).__name__})")
print(f"  gradient reached the input: {prediction2.grad is not None}")
print(f"  gradient is non-zero: {prediction2.grad.abs().sum().item() > 0}")
print(f"  gradient magnitude: {prediction2.grad.abs().mean().item():.6f}")
print("  Only this one can train a model.")

Erosion is a minimum filter, and `min(x) = −max(−x)`, so negated max-pooling implements it with an
operation autograd already differentiates. The loss is a tensor, the gradient reaches the input and
is non-zero. This is the only one of the two that can train a model.

## 3. Compactness as an alternative objective

Before soft erosion, compactness was tried: perimeter squared over area, scaled so a circle scores
1.0. It is differentiable. The question is whether it penalises the right thing.

In [ ]:
def compactness(mask_tensor):
    """Perimeter squared over area, scaled so a perfect circle scores 1.0.

    Perimeter is the summed gradient magnitude, which is differentiable, unlike a
    contour-traced perimeter.
    """
    dx = torch.abs(mask_tensor[:, :, :, 1:] - mask_tensor[:, :, :, :-1]).sum()
    dy = torch.abs(mask_tensor[:, :, 1:, :] - mask_tensor[:, :, :-1, :]).sum()
    perimeter = dx + dy
    area = mask_tensor.sum()
    return (perimeter ** 2) / (4 * np.pi * area + 1e-6)


# three shapes: a circle, an elongated lesion, and the same lesion split in two
circle = np.zeros((128, 192), np.float32)
yy, xx = np.ogrid[:128, :192]
circle[((yy - 64) ** 2 + (xx - 96) ** 2) <= 35 ** 2] = 1.0

elongated = np.zeros((128, 192), np.float32)
elongated[52:76, 40:160] = 1.0

fragmented = np.zeros((128, 192), np.float32)
fragmented[52:76, 40:100] = 1.0
fragmented[52:76, 112:160] = 1.0

shape_names = ['Circle', 'Elongated lesion', 'Fragmented lesion']
shape_data = [circle, elongated, fragmented]

print("Three shapes built, 128 x 192 each")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
compactness_scores = []

for i in range(3):
    tensor = torch.from_numpy(shape_data[i]).unsqueeze(0).unsqueeze(0)
    score = compactness(tensor).item()
    compactness_scores.append(score)
    axes[i].imshow(shape_data[i], cmap='gray')
    axes[i].set_title(f"{shape_names[i]}\ncompactness {score:.2f}", fontsize=10)
    axes[i].axis('off')

fig.suptitle('Compactness penalises elongation almost as heavily as fragmentation',
             y=1.06)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '05_compactness_result.png'), bbox_inches='tight')
plt.show()

**How to read it.** Three shapes drawn by hand, not model output, each labelled with its compactness
score. Compactness measures how much boundary a shape has for its area, so a **lower number means a
tidier shape**.

The circle is tidiest at 1.67 and the broken shape worst at 2.99 which is the behaviour wanted. The
problem is the middle one: a perfectly ordinary, single, **elongated** lesion already scores 2.29, most
of the way to the broken shape. Real lesions are elongated, so a penalty built on this would punish
correct predictions nearly as hard as broken ones. That is why the approach was set aside.

In [ ]:
print(f"{'Shape':<22} {'Area':>8} {'Compactness':>13}")
print("=" * 45)
for i in range(3):
    print(f"{shape_names[i]:<22} {int(shape_data[i].sum()):>8} "
          f"{compactness_scores[i]:>13.2f}")

Circle 1.67, elongated lesion 2.29, fragmented lesion 2.99. Fragmentation scores worst, but an
ordinary elongated lesion which is what these lesions actually look like already sits most of
the way there. The objective would penalise correct predictions for being the shape they are, which
is why it was rejected.

## 4. Models, metrics and inference

The five checkpoints of each run, the loaders and metrics, and the inference itself. These
functions are used by every section below at both input sizes, so they are written once here,
immediately above the first inference they serve.

Definitions match notebook 04, so the component counts below correspond to its CC>1 column.

In [ ]:
from architectures import (SimpleUNet, HybridUNet, OCTFSegmentation,
                        ModifiedOCTFSegmentation)
from architectureV2Res_new import ArchitectureV2

ft_model = ArchitectureV2().to(device)
ft_checkpoint = torch.load(FINETUNE_CKPT, map_location=device, weights_only=False)
ft_model.load_state_dict(ft_checkpoint['model_state_dict'])
ft_model.eval()

with open(RESIDUAL_CACHE, 'rb') as f:
    residual_cache = pickle.load(f)

# (checkpoint file, input width) for each model
MODEL_INFO = {
    'Plain U-Net':         ('plain_unet.pth', 'single'),
    'Hybrid U-Net':        ('hybrid_unet.pth', 'hybrid'),
    'OCT-F Backbone':      ('octf_backbone.pth', 'single'),
    'Modified OCT-F 2ch':  ('modified_octf_2ch.pth', 'hybrid'),
    'Soft Erosion Hybrid': ('hybrid_soft_erosion.pth', 'hybrid'),
}

models_256 = {}
models_native = {}
for run, store in [(RUN_256, models_256), (RUN_NATIVE, models_native)]:
    for name in MODEL_INFO:
        if name == 'Plain U-Net':
            model = SimpleUNet()
        elif name == 'OCT-F Backbone':
            model = OCTFSegmentation(ft_model)
        elif name == 'Modified OCT-F 2ch':
            model = ModifiedOCTFSegmentation(ft_model)
        else:
            model = HybridUNet()
        checkpoint = torch.load(os.path.join(run['ckpt_dir'], MODEL_INFO[name][0]),
                                map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        store[name] = model.to(device)

print(f"Fine-tuned OCT-F loaded from epoch {ft_checkpoint['epoch']}")
print(f"Residuals loaded: {len(residual_cache)}")
print(f"Loaded {len(models_256)} models at 256 and {len(models_native)} at native")

In [ ]:
# the same seed and sorted starting order reproduce the training split
anno_files = sorted([f for f in os.listdir(ANNO_FOLDER) if f.endswith('.tif')])
np.random.seed(SEED)
shuffled = anno_files.copy()
np.random.shuffle(shuffled)
val_files = shuffled[59:]

assert len(val_files) == 15
print(f"Validation images: {len(val_files)}")

In [ ]:
# these are used by every section below, at both input sizes, so they are
# written once here, immediately above the inference they serve
def load_gray(filename, size):
    """The endpoint scan at one run's size, as float 0 to 1."""
    height, width = size
    gray = np.array(Image.open(os.path.join(ENDPOINT_FOLDER, filename)))[:, :, 0]
    return cv2.resize(gray, (width, height)).astype(np.float32) / 255.0


def load_mask(filename, size):
    """The ground-truth mask at one run's size."""
    height, width = size
    mask = cv2.imread(os.path.join(MASK_FOLDER, filename.replace('.tif', '_mask.png')),
                      cv2.IMREAD_GRAYSCALE)
    resized = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
    return (resized > 127).astype(np.float32)


def load_residual(filename, size):
    """The cached residual at one run's size."""
    height, width = size
    return cv2.resize(residual_cache[filename], (width, height)).astype(np.float32)


def soft_erode(x, kernel_size):
    """Differentiable erosion: a minimum filter written as negated max pooling."""
    return -F.max_pool2d(-x, kernel_size=kernel_size, stride=1,
                         padding=kernel_size // 2)


def post_process(binary, kernel_size):
    """Closing to bridge nearby fragments, then keep only the largest blob."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    closed = cv2.morphologyEx(binary.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    count, labels = cv2.connectedComponents(closed)
    if count <= 1:
        return closed.astype(np.float32)

    biggest = 1
    biggest_size = 0
    for label in range(1, count):
        size = (labels == label).sum()
        if size > biggest_size:
            biggest_size = size
            biggest = label
    return (labels == biggest).astype(np.float32)


def scores(prediction, truth):
    """Dice, precision, recall and the component count for one image."""
    p = prediction.ravel()
    t = truth.ravel()
    intersection = (p * t).sum()
    dice = (2 * intersection + SMOOTH) / (p.sum() + t.sum() + SMOOTH)
    precision = (intersection + SMOOTH) / (p.sum() + SMOOTH)
    recall = (intersection + SMOOTH) / (t.sum() + SMOOTH)
    components = max(0, cv2.connectedComponents(prediction.astype(np.uint8))[0] - 1)
    return {'dice': float(dice), 'precision': float(precision),
            'recall': float(recall), 'cc': int(components)}


def evaluate_all(models, run, apply_pp):
    """Every model of one run over the 15 held-out images."""
    output = {}
    for name in models:
        store = {'dice': [], 'precision': [], 'recall': [], 'cc': [],
                 'ground_truth': [], 'prediction': [], 'probability': []}
        for filename in val_files:
            image = load_gray(filename, run['size'])
            truth = load_mask(filename, run['size'])
            if MODEL_INFO[name][1] == 'hybrid':
                stacked = np.stack([image, load_residual(filename, run['size'])])
            else:
                stacked = image[None]
            with torch.no_grad():
                probability = models[name](torch.from_numpy(stacked)[None].to(device))
            probability = probability[0, 0].cpu().numpy()
            hard = (probability > 0.5).astype(np.float32)
            if apply_pp:
                hard = post_process(hard, run['pp_kernel'])
            for key, value in scores(hard, truth).items():
                store[key].append(value)
            store['ground_truth'].append(truth)
            store['prediction'].append(hard)
            store['probability'].append(probability)
        output[name] = store
    return output


def count_fragmented(result):
    """How many of the 15 predictions broke into more than one blob."""
    total = 0
    for components in result['cc']:
        if components > 1:
            total += 1
    return total


start = time.time()
raw_256 = evaluate_all(models_256, RUN_256, apply_pp=False)
pp_256 = evaluate_all(models_256, RUN_256, apply_pp=True)
print(f"Evaluated 256x256, raw and post-processed, in {time.time() - start:.0f}s")

In [ ]:
start = time.time()
raw_native = evaluate_all(models_native, RUN_NATIVE, apply_pp=False)
pp_native = evaluate_all(models_native, RUN_NATIVE, apply_pp=True)
print(f"Evaluated native, raw and post-processed, in {time.time() - start:.0f}s")

Twenty evaluations in total: five models, two input sizes, each with and without
post-processing. Every later section reads from these four result sets rather than running
inference again.

## 5. Fragmentation incidence at 256 x 256

How often each model produces more than one blob, before any post-processing. This is the problem
the rest of the notebook is about.

In [ ]:
print(f"{'Model':<22} {'Loss':<18} {'CC>1':>6} {'Max blobs':>10} {'Dice':>8}")
print("=" * 68)
for name in MODEL_INFO:
    result = raw_256[name]
    if 'Erosion' in name:
        loss_name = 'Dice+BCE+erosion'
    else:
        loss_name = 'Dice+BCE'
    print(f"{name:<22} {loss_name:<18} {count_fragmented(result):>4}/15 "
          f"{max(result['cc']):>10} {np.mean(result['dice']):>8.3f}")

In [ ]:
total = 0
for name in MODEL_INFO:
    total += count_fragmented(raw_256[name])
print(f"Across all five models, {total} of 75 predictions at 256x256 are "
      f"multi-component before post-processing")

9 of 75 predictions are multi-component at 256. Hybrid U-Net produces none under an unconstrained
Dice+BCE objective, while Modified OCT-F 2ch produces four: fragmentation tracks the model, not
only the loss.

## 6. Penalty response to component count at 256 x 256

What the erosion penalty actually measures, on synthetic shapes of identical main-blob area with a
growing number of small fragments added.

In [ ]:
# one solid blob, then the same blob with small fragments added
shapes = {}
for n_fragments in [1, 2, 3, 5]:
    canvas = np.zeros((128, 192), np.float32)
    canvas[40:90, 60:140] = 1.0
    for k in range(n_fragments - 1):
        row = 15 + k * 12
        canvas[row:row + 6, 20:26] = 1.0
    shapes[n_fragments] = canvas

kernel = RUN_256['erosion_kernel']
fig, axes = plt.subplots(2, len(shapes), figsize=(16, 5.2))
column = 0
for n_fragments in shapes:
    canvas = shapes[n_fragments]
    tensor = torch.from_numpy(canvas).unsqueeze(0).unsqueeze(0)
    eroded = soft_erode(tensor, kernel)
    penalty = (tensor - eroded).mean().item()

    axes[0][column].imshow(canvas, cmap='gray')
    axes[0][column].set_title(f"{n_fragments} blob(s)", fontsize=11)
    axes[1][column].imshow(eroded.squeeze().numpy(), cmap='gray')
    axes[1][column].set_title(f"after erosion\npenalty {penalty:.5f}", fontsize=10)
    axes[0][column].axis('off')
    axes[1][column].axis('off')
    column += 1

fig.suptitle(f"Soft erosion with kernel {kernel}, as used at {RUN_256['label']}, "
             f"the penalty barely moves", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '05_soft_erosion_mechanics_256.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** Top row: the same large shape with an increasing number of small stray specks added: one, two, three, then five. Bottom row: what survives after erosion, which shrinks every
shape from its edges inward. The specks vanish, the main shape only slims.

The number under each column is the penalty the loss would charge. **Read across those numbers, not
the pictures** that is the whole point. They rise as specks are added, so the penalty does respond
to fragmentation, and the question the notebook asks is whether it responds *enough*.

In [ ]:
kernel = RUN_256['erosion_kernel']
penalties = {}
for n_fragments in shapes:
    tensor = torch.from_numpy(shapes[n_fragments]).unsqueeze(0).unsqueeze(0)
    penalties[n_fragments] = (tensor - soft_erode(tensor, kernel)).mean().item()

baseline = penalties[1]
print(f"Kernel {kernel}")
print(f"{'Fragments':>10} {'Penalty':>12} {'Increase':>12}")
print("=" * 36)
for n_fragments in penalties:
    penalty = penalties[n_fragments]
    if n_fragments == 1:
        increase = '-'
    else:
        increase = f"{(penalty - baseline) / baseline * 100:+.1f}%"
    print(f"{n_fragments:>10} {penalty:>12.6f} {increase:>12}")

Penalty rises from 0.020508 to 0.025716 between one and five components, **+25.4%** for four
extra fragments. The signal is in the right direction but small, and it is weighted at 0.2 against
0.4 for Dice and 0.4 for BCE.

## 7. Penalty behaviour on an observed prediction at 256 x 256

The synthetic case is favourable. This applies the same erosion to a real fragmented probability
map from the model that fragments most.

In [ ]:
# the model that fragments most, on its first fragmented image
worst_name = ''
most = -1
for name in MODEL_INFO:
    multi = count_fragmented(raw_256[name])
    if multi > most:
        most = multi
        worst_name = name

index = 0
for i in range(len(val_files)):
    if raw_256[worst_name]['cc'][i] > 1:
        index = i
        break

probability = raw_256[worst_name]['probability'][index]
tensor = torch.from_numpy(probability).unsqueeze(0).unsqueeze(0)
eroded = soft_erode(tensor, RUN_256['erosion_kernel']).squeeze().numpy()
removed = probability - eroded

fig, axes = plt.subplots(1, 4, figsize=(19, 3.4))
axes[0].imshow(raw_256[worst_name]['ground_truth'][index], cmap='gray')
axes[0].set_title('Ground truth', fontsize=10)
axes[1].imshow(raw_256[worst_name]['prediction'][index], cmap='gray')
axes[1].set_title(f"Prediction: {raw_256[worst_name]['cc'][index]} components",
                  fontsize=10)
axes[2].imshow(eroded, cmap='gray')
axes[2].set_title('After soft erosion', fontsize=10)
axes[3].imshow(removed, cmap='inferno', vmin=0, vmax=0.5)
axes[3].set_title(f"Removed by erosion (mean {removed.mean():.4f})", fontsize=10)

for ax in axes:
    ax.axis('off')

fig.suptitle(f"Erosion applied to a finished prediction, not a model trained with it: {worst_name} on {val_files[index][:-4]} at {RUN_256['label']}", y=1.05)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '05_erosion_effect_256.png'), bbox_inches='tight')
plt.show()

**How to read it.** Four panels: what the expert marked, what the model predicted, what remains after
erosion, and in yellow the outline of what erosion removed.

The comparison that matters is the **component count in the second and third titles**. If erosion
worked as intended, the stray piece would disappear and the count would fall. Watch whether it does: a fragment wider than the erosion kernel survives being shrunk.

In [ ]:
before = cv2.connectedComponents((probability > 0.5).astype(np.uint8))[0] - 1
after = cv2.connectedComponents((eroded > 0.5).astype(np.uint8))[0] - 1
print(f"{worst_name} on {val_files[index][:-4]} at {RUN_256['label']}")
print(f"  Components before erosion: {before}")
print(f"  Components after erosion:  {after}")

Component count falls from 3 to 2. Erosion thins fragments rather than eliminating them; any
fragment wider than the kernel survives. The penalty is a pressure on fragment size, not on
fragment count.

## 8. Controlled comparison of the objective at 256 x 256

Hybrid U-Net against Soft Erosion Hybrid. Same architecture, same data, same split, **zero** parameters between them: the loss function is the only difference, so any difference in
fragmentation is attributable to the objective.

In [ ]:
# the two models are the same architecture with zero parameters between them,
# so the loss function is the only difference
PAIR = ['Hybrid U-Net', 'Soft Erosion Hybrid']

print(f"{'Model':<22} {'Loss':<20} {'CC>1':>6} {'Dice':>8} {'Prec':>7} {'Recall':>8}")
print("=" * 74)
for name in PAIR:
    result = raw_256[name]
    if 'Erosion' in name:
        loss_name = 'Dice+BCE+erosion'
    else:
        loss_name = 'Dice+BCE'
    print(f"{name:<22} {loss_name:<20} {count_fragmented(result):>4}/15 "
          f"{np.mean(result['dice']):>8.3f} {np.mean(result['precision']):>7.3f} "
          f"{np.mean(result['recall']):>8.3f}")

In [ ]:
print(f"{'Image':<22} {'Hybrid':>8} {'SoftEro':>9}")
print("-" * 41)
for i in range(len(val_files)):
    print(f"{val_files[i][:-4]:<22} {raw_256[PAIR[0]]['cc'][i]:>8} "
          f"{raw_256[PAIR[1]]['cc'][i]:>9}")

In [ ]:
# the first image where the two disagree on the component count
index = 0
for i in range(len(val_files)):
    if raw_256[PAIR[0]]['cc'][i] != raw_256[PAIR[1]]['cc'][i]:
        index = i
        break

gray = (load_gray(val_files[index], RUN_256['size']) * 255).astype(np.uint8)
ground_truth = raw_256[PAIR[0]]['ground_truth'][index]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
axes[0].imshow(gray, cmap='gray')
axes[0].contour(ground_truth, levels=[0.5], colors='#39FF6A', linewidths=1.2)
axes[0].set_title(f"{val_files[index][:-4]} (green = truth)", fontsize=10)

position = 1
for name in PAIR:
    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    overlay[raw_256[name]['prediction'][index] > 0] = PRED_RGB
    axes[position].imshow(overlay)
    axes[position].contour(ground_truth, levels=[0.5], colors='#39FF6A',
                           linewidths=1.2)
    axes[position].set_title(f"{name}\n{raw_256[name]['cc'][index]} component(s), "
                             f"Dice {raw_256[name]['dice'][index]:.3f}", fontsize=10)
    position += 1

for ax in axes:
    ax.axis('off')

fig.suptitle(f"Same architecture, zero parameters apart, different loss, "
             f"at {RUN_256['label']}", y=1.06)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '05_soft_erosion_result_256.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** Three panels of the same scan. The **green outline** is the expert's lesion in all
of them; the **red fill** is what the model predicted. The left panel is the truth alone, then two
models.

Those two models are **architecturally identical, zero parameters apart**, and differ only in the
loss they were trained with. So any difference between the middle and right panels is caused by the
loss function and nothing else. Read the component counts and Dice in the titles together.

The model carrying the connectivity penalty returns **1** multi-component prediction against
**0** for the unconstrained control. The pair disagrees on one of 15 images, with the penalised
model producing two components where the control produced one the opposite of the intended
effect.

## 9. Constraint placement at 256 x 256

The four ways the constraint can be applied, on the same architecture: not at all, in the loss,
after training, or both.

In [ ]:
# the four ways the single-region constraint can be applied, in the order the
# tables below use. two of them are the same model read with and without PP
APPROACH_NAMES = ['No constraint (Dice+BCE)',
                  'Erosion penalty in the loss',
                  'Post-processing after training',
                  'Erosion penalty + post-processing']
DIFFERENTIABLE = ['n/a', 'yes', 'no', 'partly']

for i in range(4):
    print(f"{APPROACH_NAMES[i]:<34} differentiable: {DIFFERENTIABLE[i]}")

In [ ]:
chosen = [raw_256['Hybrid U-Net'], raw_256['Soft Erosion Hybrid'],
          pp_256['Hybrid U-Net'], pp_256['Soft Erosion Hybrid']]

print(f"{'Approach':<34} {'CC>1':>7} {'Dice':>8} {'Differentiable':>15}")
print("=" * 68)
for i in range(4):
    print(f"{APPROACH_NAMES[i]:<34} {count_fragmented(chosen[i]):>5}/15 "
          f"{np.mean(chosen[i]['dice']):>8.3f} {DIFFERENTIABLE[i]:>15}")

In [ ]:
short_names = ['No\nconstraint', 'Erosion\nloss', 'Post-\nprocessing', 'Erosion\n+ PP']
colours = ['#8A8A8A', '#C0392B', '#2F6FA8', '#2E9E5B']

multi_counts = []
dice_means = []
for result in chosen:
    multi_counts.append(count_fragmented(result))
    dice_means.append(np.mean(result['dice']))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
bars = axes[0].bar(short_names, multi_counts, color=colours, edgecolor='white')
for bar, value in zip(bars, multi_counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, value + 0.05, str(value),
                 ha='center', fontweight='bold')
axes[0].set_ylabel('images with more than one blob (of 15)')
axes[0].set_title('Fragmentation: lower is better')

bars = axes[1].bar(short_names, dice_means, color=colours, edgecolor='white')
for bar, value in zip(bars, dice_means):
    axes[1].text(bar.get_x() + bar.get_width() / 2, value + 0.004,
                 f"{value:.3f}", ha='center', fontweight='bold')
axes[1].set_ylabel('mean Dice')
axes[1].set_ylim(0.7, 0.9)
axes[1].set_title('Segmentation quality: barely moves')

fig.suptitle(f"Connectivity approaches at {RUN_256['label']}", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '05_connectivity_comparison_256.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** Four approaches to keeping a prediction in one piece, drawn twice. The **left**
panel counts how many of the 15 images came out broken, **lower is better**. The **right** panel is
segmentation quality on the same runs, and note its axis starts near 0.70 rather than 0, so small
differences look larger than they are.

Read the two panels together. The honest summary is that only post-processing drives fragmentation to
zero, while quality on the right barely moves whichever approach is used.

Penalty in the objective: 1/15 multi-component. Unconstrained: 0/15. Post-processing: 0/15 at a
cost of 0.000 Dice. The constraint applied after training achieves what the constraint inside the
loss does not.

## 10. Fragmentation incidence at native 496 x 768

The same measurement at the acquisition geometry, written out again so this half of the notebook
can be read on its own.

In [ ]:
print(f"{'Model':<22} {'Loss':<18} {'CC>1':>6} {'Max blobs':>10} {'Dice':>8}")
print("=" * 68)
for name in MODEL_INFO:
    result = raw_native[name]
    if 'Erosion' in name:
        loss_name = 'Dice+BCE+erosion'
    else:
        loss_name = 'Dice+BCE'
    print(f"{name:<22} {loss_name:<18} {count_fragmented(result):>4}/15 "
          f"{max(result['cc']):>10} {np.mean(result['dice']):>8.3f}")

In [ ]:
total = 0
for name in MODEL_INFO:
    total += count_fragmented(raw_native[name])
print(f"Across all five models, {total} of 75 predictions at native 496x768 are "
      f"multi-component before post-processing")

24 of 75 at native against 9 of 75 at 256: **2.7 times** the incidence. Modified OCT-F 2ch is
worst at 10 of 15. Fragmentation is substantially more common at the larger input.

## 11. Penalty response at the larger kernel

The same synthetic experiment with the kernel that native uses. The kernel scales with the image,
and that changes what the penalty is sensitive to.

In [ ]:
# one solid blob, then the same blob with small fragments added
shapes = {}
for n_fragments in [1, 2, 3, 5]:
    canvas = np.zeros((128, 192), np.float32)
    canvas[40:90, 60:140] = 1.0
    for k in range(n_fragments - 1):
        row = 15 + k * 12
        canvas[row:row + 6, 20:26] = 1.0
    shapes[n_fragments] = canvas

kernel = RUN_NATIVE['erosion_kernel']
fig, axes = plt.subplots(2, len(shapes), figsize=(16, 5.2))
column = 0
for n_fragments in shapes:
    canvas = shapes[n_fragments]
    tensor = torch.from_numpy(canvas).unsqueeze(0).unsqueeze(0)
    eroded = soft_erode(tensor, kernel)
    penalty = (tensor - eroded).mean().item()

    axes[0][column].imshow(canvas, cmap='gray')
    axes[0][column].set_title(f"{n_fragments} blob(s)", fontsize=11)
    axes[1][column].imshow(eroded.squeeze().numpy(), cmap='gray')
    axes[1][column].set_title(f"after erosion\npenalty {penalty:.5f}", fontsize=10)
    axes[0][column].axis('off')
    axes[1][column].axis('off')
    column += 1

fig.suptitle(f"Soft erosion with kernel {kernel}, as used at {RUN_NATIVE['label']}, "
             f"the penalty barely moves", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '05_soft_erosion_mechanics.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** Top row: one large shape with an increasing number of small specks added.
Bottom row: what is left after erosion eats each shape inward from its edges.

**Compare the four penalty numbers.** Going from one speck to five moves the penalty by only about
15%. With the larger kernel used at this input size, specks smaller than the kernel are erased
completely, so adding more of them costs almost nothing. That is the mechanism behind the finding
that the penalty is weaker at native geometry, and the title says it plainly: the penalty barely moves.

In [ ]:
kernel = RUN_NATIVE['erosion_kernel']
penalties = {}
for n_fragments in shapes:
    tensor = torch.from_numpy(shapes[n_fragments]).unsqueeze(0).unsqueeze(0)
    penalties[n_fragments] = (tensor - soft_erode(tensor, kernel)).mean().item()

baseline = penalties[1]
print(f"Kernel {kernel}")
print(f"{'Fragments':>10} {'Penalty':>12} {'Increase':>12}")
print("=" * 36)
for n_fragments in penalties:
    penalty = penalties[n_fragments]
    if n_fragments == 1:
        increase = '-'
    else:
        increase = f"{(penalty - baseline) / baseline * 100:+.1f}%"
    print(f"{n_fragments:>10} {penalty:>12.6f} {increase:>12}")

Baseline penalty 0.039714 at kernel 9 against 0.020508 at kernel 5, since a larger kernel erodes
more of the main blob. The rise for four extra fragments is **+14.8%** against +25.4% at 256:
fragments are a smaller share of a larger signal, so the penalty is proportionally weaker exactly
where fragmentation is worse.

## 12. Penalty behaviour on an observed prediction at native

The same check on a real native prediction, where the kernel is 9 rather than 5.

In [ ]:
# the model that fragments most, on its first fragmented image
worst_name = ''
most = -1
for name in MODEL_INFO:
    multi = count_fragmented(raw_native[name])
    if multi > most:
        most = multi
        worst_name = name

index = 0
for i in range(len(val_files)):
    if raw_native[worst_name]['cc'][i] > 1:
        index = i
        break

probability = raw_native[worst_name]['probability'][index]
tensor = torch.from_numpy(probability).unsqueeze(0).unsqueeze(0)
eroded = soft_erode(tensor, RUN_NATIVE['erosion_kernel']).squeeze().numpy()
removed = probability - eroded

fig, axes = plt.subplots(1, 4, figsize=(19, 3.4))
axes[0].imshow(raw_native[worst_name]['ground_truth'][index], cmap='gray')
axes[0].set_title('Ground truth', fontsize=10)
axes[1].imshow(raw_native[worst_name]['prediction'][index], cmap='gray')
axes[1].set_title(f"Prediction: {raw_native[worst_name]['cc'][index]} components",
                  fontsize=10)
axes[2].imshow(eroded, cmap='gray')
axes[2].set_title('After soft erosion', fontsize=10)
axes[3].imshow(removed, cmap='inferno', vmin=0, vmax=0.5)
axes[3].set_title(f"Removed by erosion (mean {removed.mean():.4f})", fontsize=10)

for ax in axes:
    ax.axis('off')

fig.suptitle(f"Erosion applied to a finished prediction, not a model trained with it: {worst_name} on {val_files[index][:-4]} at {RUN_NATIVE['label']}", y=1.05)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '05_erosion_effect.png'), bbox_inches='tight')
plt.show()

**How to read it.** Left to right: the expert's mask, the model's prediction with its component count,
the prediction after erosion with its count, and the region erosion stripped away outlined in yellow.

**The counts in the two middle titles are the result.** Here the prediction has two components before
erosion and still has two after, because the stray piece is wider than the erosion kernel and so is
only slimmed, never removed. Erosion takes a rim off everything rather than deleting fragments.

In [ ]:
before = cv2.connectedComponents((probability > 0.5).astype(np.uint8))[0] - 1
after = cv2.connectedComponents((eroded > 0.5).astype(np.uint8))[0] - 1
print(f"{worst_name} on {val_files[index][:-4]} at {RUN_NATIVE['label']}")
print(f"  Components before erosion: {before}")
print(f"  Components after erosion:  {after}")

Component count is unchanged at 2. The fragment exceeds the 9x9 kernel in width and survives
erosion intact, so the penalty cannot register it. At native the penalty is both proportionally
weaker and less able to reach the fragments that exist.

## 13. Controlled comparison of the objective at native

The same zero-parameter pair at native geometry.

In [ ]:
# the two models are the same architecture with zero parameters between them,
# so the loss function is the only difference
PAIR = ['Hybrid U-Net', 'Soft Erosion Hybrid']

print(f"{'Model':<22} {'Loss':<20} {'CC>1':>6} {'Dice':>8} {'Prec':>7} {'Recall':>8}")
print("=" * 74)
for name in PAIR:
    result = raw_native[name]
    if 'Erosion' in name:
        loss_name = 'Dice+BCE+erosion'
    else:
        loss_name = 'Dice+BCE'
    print(f"{name:<22} {loss_name:<20} {count_fragmented(result):>4}/15 "
          f"{np.mean(result['dice']):>8.3f} {np.mean(result['precision']):>7.3f} "
          f"{np.mean(result['recall']):>8.3f}")

In [ ]:
print(f"{'Image':<22} {'Hybrid':>8} {'SoftEro':>9}")
print("-" * 41)
for i in range(len(val_files)):
    print(f"{val_files[i][:-4]:<22} {raw_native[PAIR[0]]['cc'][i]:>8} "
          f"{raw_native[PAIR[1]]['cc'][i]:>9}")

In [ ]:
# the first image where the two disagree on the component count
index = 0
for i in range(len(val_files)):
    if raw_native[PAIR[0]]['cc'][i] != raw_native[PAIR[1]]['cc'][i]:
        index = i
        break

gray = (load_gray(val_files[index], RUN_NATIVE['size']) * 255).astype(np.uint8)
ground_truth = raw_native[PAIR[0]]['ground_truth'][index]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
axes[0].imshow(gray, cmap='gray')
axes[0].contour(ground_truth, levels=[0.5], colors='#39FF6A', linewidths=1.2)
axes[0].set_title(f"{val_files[index][:-4]} (green = truth)", fontsize=10)

position = 1
for name in PAIR:
    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    overlay[raw_native[name]['prediction'][index] > 0] = PRED_RGB
    axes[position].imshow(overlay)
    axes[position].contour(ground_truth, levels=[0.5], colors='#39FF6A',
                           linewidths=1.2)
    axes[position].set_title(f"{name}\n{raw_native[name]['cc'][index]} component(s), "
                             f"Dice {raw_native[name]['dice'][index]:.3f}", fontsize=10)
    position += 1

for ax in axes:
    ax.axis('off')

fig.suptitle(f"Same architecture, zero parameters apart, different loss, "
             f"at {RUN_NATIVE['label']}", y=1.06)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '05_soft_erosion_result.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** Green outline is the expert's lesion, red fill is the prediction. Left is truth
alone; middle and right are two models that are **identical in architecture and parameter count** and
differ only in their loss function.

That makes this a clean controlled comparison. The titles give component count and Dice for each. The
model trained *with* the fragment penalty ends up with **more** components than the one without it,
at essentially the same Dice, which is the opposite of what the penalty was added to achieve.

The penalised model returns **3** multi-component predictions against **2** for the control. The
penalty adds a fragmented image at this input size too, so the result at 256 was not a one-off.

## 14. Constraint placement at native

The same four approaches at native geometry.

In [ ]:
chosen = [raw_native['Hybrid U-Net'], raw_native['Soft Erosion Hybrid'],
          pp_native['Hybrid U-Net'], pp_native['Soft Erosion Hybrid']]

print(f"{'Approach':<34} {'CC>1':>7} {'Dice':>8} {'Differentiable':>15}")
print("=" * 68)
for i in range(4):
    print(f"{APPROACH_NAMES[i]:<34} {count_fragmented(chosen[i]):>5}/15 "
          f"{np.mean(chosen[i]['dice']):>8.3f} {DIFFERENTIABLE[i]:>15}")

In [ ]:
short_names = ['No\nconstraint', 'Erosion\nloss', 'Post-\nprocessing', 'Erosion\n+ PP']
colours = ['#8A8A8A', '#C0392B', '#2F6FA8', '#2E9E5B']

multi_counts = []
dice_means = []
for result in chosen:
    multi_counts.append(count_fragmented(result))
    dice_means.append(np.mean(result['dice']))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
bars = axes[0].bar(short_names, multi_counts, color=colours, edgecolor='white')
for bar, value in zip(bars, multi_counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, value + 0.05, str(value),
                 ha='center', fontweight='bold')
axes[0].set_ylabel('images with more than one blob (of 15)')
axes[0].set_title('Fragmentation: lower is better')

bars = axes[1].bar(short_names, dice_means, color=colours, edgecolor='white')
for bar, value in zip(bars, dice_means):
    axes[1].text(bar.get_x() + bar.get_width() / 2, value + 0.004,
                 f"{value:.3f}", ha='center', fontweight='bold')
axes[1].set_ylabel('mean Dice')
axes[1].set_ylim(0.7, 0.9)
axes[1].set_title('Segmentation quality: barely moves')

fig.suptitle(f"Connectivity approaches at {RUN_NATIVE['label']}", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '05_connectivity_comparison.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** Four approaches, two questions. **Left:** how many of the 15 images came out in more than one piece, lower is better. **Right:** the segmentation quality that came with it.

Two things to say. Post-processing is the only approach that reaches zero, and the erosion penalty
actually produces **more** broken images than using no constraint at all. Meanwhile the right panel
hardly moves: **its axis starts near 0.70, so the visible differences are much smaller than they
appear**, spanning about four thousandths of a point.

The penalty increases multi-component incidence by one image; post-processing yields 0/15 at a
cost of 0.001 Dice. Both halves of the notebook reach the same conclusion independently.

## 15. Fragmentation across both runs

Every model at both input sizes, before and after post-processing, in one table.

In [ ]:
print(f"{'Model':<22} {'256 raw':>8} {'Nat raw':>8} {'256 PP':>8} {'Nat PP':>8}")
print("=" * 58)
for name in MODEL_INFO:
    print(f"{name:<22} {count_fragmented(raw_256[name]):>6}/15 "
          f"{count_fragmented(raw_native[name]):>6}/15 "
          f"{count_fragmented(pp_256[name]):>6}/15 "
          f"{count_fragmented(pp_native[name]):>6}/15")

total_256 = 0
total_native = 0
for name in MODEL_INFO:
    total_256 += count_fragmented(raw_256[name])
    total_native += count_fragmented(raw_native[name])
print()
print(f"Total before post-processing: {total_256}/75 at 256, "
      f"{total_native}/75 at native")

In [ ]:
names = []
counts_256 = []
counts_native = []
for name in MODEL_INFO:
    names.append(name.replace(' ', '\n', 1))
    counts_256.append(count_fragmented(raw_256[name]))
    counts_native.append(count_fragmented(raw_native[name]))

x = np.arange(len(names))
width = 0.38

fig, ax = plt.subplots(figsize=(12, 4.4))
ax.bar(x - width / 2, counts_256, width, label='256 x 256', color='#2F6FA8',
       edgecolor='white')
ax.bar(x + width / 2, counts_native, width, label='496 x 768', color='#C0392B',
       edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel('images with more than one blob (of 15)')
ax.set_title('Fragmentation before post-processing: worse at every model at native')
ax.legend()
ax.grid(alpha=0.25, axis='y')

fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '05_fragmentation_comparison.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** One pair of bars per model: blue is the smaller input size, red is full size. Bar
height is how many of the 15 images came out in more than one piece, before any clean-up.

**Red is taller than blue for every model**, so fragmentation is worse at full resolution across the board: a bigger image gives the model more room to leave stray specks. One model stands out with ten
of fifteen images broken. After post-processing every one of these bars would be zero.

Incidence is higher at native for every model, and rank order is preserved across input sizes:
Modified OCT-F 2ch highest, Hybrid U-Net lowest. Post-processing yields **0/15 in all ten
configurations**.

## 16. Constraint placement, both runs

The four approaches side by side at both input sizes: the table this notebook exists to produce.

In [ ]:
chosen_256 = [raw_256['Hybrid U-Net'], raw_256['Soft Erosion Hybrid'],
              pp_256['Hybrid U-Net'], pp_256['Soft Erosion Hybrid']]
chosen_native = [raw_native['Hybrid U-Net'], raw_native['Soft Erosion Hybrid'],
                 pp_native['Hybrid U-Net'], pp_native['Soft Erosion Hybrid']]

print(f"{'Approach':<34} {'256 CC>1':>9} {'Nat CC>1':>9} {'256 Dice':>9} {'Nat Dice':>9}")
print("=" * 74)
for i in range(4):
    print(f"{APPROACH_NAMES[i]:<34} {count_fragmented(chosen_256[i]):>7}/15 "
          f"{count_fragmented(chosen_native[i]):>7}/15 "
          f"{np.mean(chosen_256[i]['dice']):>9.3f} "
          f"{np.mean(chosen_native[i]['dice']):>9.3f}")

The penalty increases multi-component incidence by one image at **both** input sizes;
post-processing removes every fragmented prediction at both. Two independent runs, the same
answer: the single-region constraint belongs after training rather than inside the objective.

## 17. Rejected approaches

The four objectives tried and set aside, with the reason each failed. Two of the numbers here come
from earlier 256x256 experiments and are not reproduced by the current protocol.

In [ ]:
print("Largest-component loss")
print("  Not differentiable. OpenCV detaches the tensor from the graph, so no gradient")
print("  reaches the model. Cannot be implemented as a loss at all.")
print()
print("Compactness loss, perimeter squared over area")
print("  Differentiable, but it penalises all boundary. Real lesions are elongated, so it")
print("  fights the ground truth. Earlier experiment: Dice fell from 0.803 to 0.515.")
print()
print("Soft erosion penalty")
print("  Differentiable and specific to fragments, but targets fragment size rather")
print("  than count. Kept as model 5, and it adds a fragmented image at both sizes.")
print()
print("1x1 projection for the 2-channel OCT-F")
print("  Compressing image and residual into one channel before the pretrained encoder")
print("  destroyed the residual signal. Earlier experiment: Dice 0.471.")
print()
print("Larger decoder, 3.61M parameters")
print("  No improvement over the 0.76M decoder. With 59 training images the data is the")
print("  limiting factor, not decoder capacity.")

Two values here, the compactness result and the 1x1 projection Dice, originate from earlier
experiments and are recorded in `docs/NUMBERS.md` as history rather than as current results.

## 18. Output manifest

Every figure this notebook writes, for the figure catalogue in the supporting material.

In [ ]:
print("Figures saved in outputs/images/:")
for name in sorted(os.listdir(IMAGES_DIR)):
    if name.startswith('05_'):
        print(f"  {name}")

---

## Summary of notebook 05

**Where does the single-region constraint belong: in the loss, or after training?**

A component count cannot be a loss. `cv2.connectedComponents` returns a plain float, so
`.backward()` has nothing to act on. Soft erosion is the differentiable stand-in: written as
negated max pooling it returns a tensor and its gradient reaches the input.

**Compactness was rejected first.** Circle 1.67, elongated lesion 2.29, fragmented lesion 2.99.
Fragmentation scores worst, but an ordinary elongated lesion, which is what these lesions look like, is already penalised most of the way there, so the objective fights the ground truth.

**The soft erosion penalty is real but weak.** Between one and five fragments it moves from
0.020508 to 0.025716 at kernel 5, **+25.4%**, and from 0.039714 to 0.045573 at kernel 9,
**+14.8%**. The kernel-9 baseline is nearly double the kernel-5 baseline because a larger kernel
erodes more of the main blob, so fragments are a smaller share of the signal at native, which is a mechanism for why the penalty is weaker there.

**On a real prediction it thins fragments rather than removing them.** At 256 the component count
falls from 3 to 2; at native it stays at 2, because the fragment is wider than the 9x9 kernel and
survives unchanged.

**The controlled comparison is the result.** Hybrid U-Net and Soft Erosion Hybrid are the same
architecture with **zero** parameters between them. The model carrying the penalty returns **more**
fragmented predictions, not fewer: 1 against 0 at 256, and 3 against 2 at native. The same
conclusion from two independent runs.

**Post-processing reaches 0/15 in every configuration**, at both input sizes, at a cost of at most
0.005 Dice.

**The constraint belongs after training, not in the objective.** Fragmentation is also 2.7 times
more common at native (24 of 75) than at 256 (9 of 75), so the input geometry matters more to
connectivity than the loss function does.

**Next:** notebook 04 reports the full evaluation these predictions come from.